## 1. Тест на первой странице
Проверяем базовую логику парсинга, собираем ссылки с первой страницы листинга.

In [2]:
import requests
from bs4 import BeautifulSoup
url = "https://krisha.kz/prodazha/kvartiry/"
headers = {"User-agent": "Mozilla/5.0"}
response = requests.get(url, headers = headers)
soup = BeautifulSoup(response.content, 'html.parser')

list_of_links = []

cards = soup.findAll("div", attrs = {'class': 'a-card__inc'})
for card in cards:
    a_tag = card.find("a")
    if a_tag and a_tag.get("href"):
        link = "https://krisha.kz" + a_tag["href"]
        list_of_links.append(link)
print(list_of_links)

['https://krisha.kz/a/show/1012178700', 'https://krisha.kz/a/show/698590467', 'https://krisha.kz/a/show/694170572', 'https://krisha.kz/a/show/1012268913', 'https://krisha.kz/a/show/1010804699', 'https://krisha.kz/a/show/697546174', 'https://krisha.kz/a/show/691064914', 'https://krisha.kz/a/show/1011375260', 'https://krisha.kz/a/show/1007113061', 'https://krisha.kz/a/show/688070600', 'https://krisha.kz/a/show/1011768173', 'https://krisha.kz/a/show/1011619627', 'https://krisha.kz/a/show/1011582608', 'https://krisha.kz/a/show/1011485943', 'https://krisha.kz/a/show/1011047608', 'https://krisha.kz/a/show/1010808248', 'https://krisha.kz/a/show/1010704207', 'https://krisha.kz/a/show/1010375317', 'https://krisha.kz/a/show/1009011409', 'https://krisha.kz/a/show/1008955050', 'https://krisha.kz/a/show/1007548684', 'https://krisha.kz/a/show/1006368202', 'https://krisha.kz/a/show/1006042603']


## 2. Сбор ссылок, страницы 1–300
Обходим первые 300 страниц со случайными задержками (2–5 сек). Сохраняем в `links_part1.txt`.

In [6]:
import time
import random

base_url = "https://krisha.kz/prodazha/kvartiry/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8",
    "Connection": "keep-alive",
}

all_links = []

# Собираем ссылки со всех страниц
urls = [base_url] + [base_url + f"?page={p}" for p in range(2, 301)]

for url in urls:
    print(f"Скрапинг: {url}")
    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f" Ошибка {response.status_code}, пропускаем")
        continue

    soup = BeautifulSoup(response.content, 'html.parser')
    cards = soup.findAll("div", attrs={'class': 'a-card__inc'})
    if not cards:
        print("Страницы закончились, стоп!")
        break

    page_links = []
    for card in cards:
        a_tag = card.find("a")
        if a_tag and a_tag.get("href"):
            link = "https://krisha.kz" + a_tag["href"]
            page_links.append(link)

    all_links.extend(page_links)
    print(f" Найдено: {len(page_links)} ссылок")

    sleep_time = random.uniform(2, 5)
    print(f" Ждём {sleep_time:.1f} сек...")
    time.sleep(sleep_time)

print(f"\nИтого ссылок: {len(all_links)}")

# В конце каждого запуска сохранить в файл
with open("links_part1.txt", "w") as f:
    for link in all_links:
        f.write(link + "\n")

Скрапинг: https://krisha.kz/prodazha/kvartiry/
 Найдено: 23 ссылок
 Ждём 4.0 сек...
  ... (ещё 896 строк) ...
 Ждём 4.3 сек...

Итого ссылок: 6003


## 3. Сбор ссылок, страницы 301–600
Продолжаем сбор. Результат сохраняется в `links_part2.txt`.

In [10]:
all_links = []

urls = [base_url + f"?page={p}" for p in range(301, 601)]

for url in urls:
    print(f"Скрапинг: {url}")
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f" Ошибка {response.status_code}, пропускаем")
        continue
        
    soup = BeautifulSoup(response.content, 'html.parser')
    cards = soup.findAll("div", attrs={'class': 'a-card__inc'})
    if not cards:
        print("Страницы закончились, стоп!")
        break
        
    page_links = []
    for card in cards:
        a_tag = card.find("a")
        if a_tag and a_tag.get("href"):
            link = "https://krisha.kz" + a_tag["href"]
            page_links.append(link)
            
    all_links.extend(page_links)
    print(f" Найдено: {len(page_links)} ссылок")
    
    sleep_time = random.uniform(2, 5)
    print(f" Ждём {sleep_time:.1f} сек...")
    time.sleep(sleep_time)
    
print(f"\nИтого ссылок: {len(all_links)}")

with open("links_part2.txt", "w") as f:
    for link in all_links:
        f.write(link + "\n")

Скрапинг: https://krisha.kz/prodazha/kvartiry/?page=301
 Найдено: 20 ссылок
 Ждём 2.1 сек...
  ... (ещё 896 строк) ...
 Ждём 2.4 сек...

Итого ссылок: 6000


## 4. Сбор ссылок, страницы 601–950
Финальный батч. Если страницы закончатся раньше, `break` остановит loop. Результат в `links_part3.txt`.

In [12]:
all_links = []

urls = [base_url + f"?page={p}" for p in range(601, 951)]

for url in urls:
    print(f"Скрапинг: {url}")
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f" Ошибка {response.status_code}, пропускаем")
        continue
        
    soup = BeautifulSoup(response.content, 'html.parser')
    cards = soup.findAll("div", attrs={'class': 'a-card__inc'})
    if not cards:
        print("Страницы закончились, стоп!")
        break
        
    page_links = []
    for card in cards:
        a_tag = card.find("a")
        if a_tag and a_tag.get("href"):
            link = "https://krisha.kz" + a_tag["href"]
            page_links.append(link)
            
    all_links.extend(page_links)
    print(f" Найдено: {len(page_links)} ссылок")
    
    sleep_time = random.uniform(2, 5)
    print(f" Ждём {sleep_time:.1f} сек...")
    time.sleep(sleep_time)
    
print(f"\nИтого ссылок: {len(all_links)}")
with open("links_part3.txt", "w") as f:
    for link in all_links:
        f.write(link + "\n")

Скрапинг: https://krisha.kz/prodazha/kvartiry/?page=601
 Найдено: 20 ссылок
 Ждём 4.5 сек...
  ... (ещё 1046 строк) ...
 Ждём 2.8 сек...

Итого ссылок: 7000


## 5. Объединение всех ссылок
Сливаем три файла в один список `all_links_combined`.

In [14]:
all_links_combined = []

for filename in ["links_part1.txt", "links_part2.txt", "links_part3.txt"]:
    with open(filename, "r") as f:
        links = f.read().splitlines()
        all_links_combined.extend(links)

print(f"Всего ссылок: {len(all_links_combined)}")

Всего ссылок: 19003


## 6. Дедупликация
Удаляем дубликаты через `set()` -- они появляются из-за новых объявлений, которые сдвигают пагинацию во время сбора.

In [16]:
print(f"До удаления дубликатов: {len(all_links_combined)}")

all_links_combined = list(set(all_links_combined))

print(f"После удаления дубликатов: {len(all_links_combined)}")

До удаления дубликатов: 19003
После удаления дубликатов: 17375


## 7. Просмотр первых ссылок
Быстрая проверка что данные выглядят корректно.

In [22]:
all_links_combined[0:10]

['https://krisha.kz/a/show/1012063292',
 'https://krisha.kz/a/show/1011939022',
 'https://krisha.kz/a/show/1011458594',
 'https://krisha.kz/a/show/1011710667',
 'https://krisha.kz/a/show/1012269199',
 'https://krisha.kz/a/show/1012271793',
 'https://krisha.kz/a/show/1012150112',
 'https://krisha.kz/a/show/1003362629',
 'https://krisha.kz/a/show/689196236',
 'https://krisha.kz/a/show/1010846608']

## 8. Функция парсинга одного объявления
Извлекает: цену, количество комнат, город, тип дома, год постройки, этаж, площадь и все дополнительные атрибуты из `dl`-блоков.

In [30]:
import re

def parse_listing(url, soup):
    data = {"url": url}

    # Заголовок
    h1 = soup.find("h1")
    data["заголовок"] = h1.text.strip() if h1 else None

    # Комнаты из заголовка
    if data["заголовок"]:
        rooms_match = re.search(r"(\d+)-комнатная", data["заголовок"])
        data["комнаты"] = int(rooms_match.group(1)) if rooms_match else None
    else:
        data["комнаты"] = None

    # Цена
    
    # Цена - сначала из div, потом из title страницы
    price_div = soup.find("div", class_="offer__price")
    if price_div:
        price_raw = price_div.text.strip()
        data["цена_raw"] = price_raw
        digits = re.sub(r"[^\d]", "", price_raw)
        data["цена"] = int(digits) if digits else None
    else:
        # fallback — берём из <title> тега
        title_tag = soup.find("title")
        if title_tag:
            price_match = re.search(r"за (\d+)", title_tag.text)
            data["цена"] = int(price_match.group(1)) if price_match else None
            data["цена_raw"] = f"из title: {data['цена']}"
        else:
            data["цена"] = None
            data["цена_raw"] = None

    # Sidebar - город, тип дома, год, этаж, площадь и тд
    for item in soup.findAll("div", class_="offer__info-item"):
        name = item.get("data-name")
        value_div = item.find("div", class_="offer__advert-short-info")
        if value_div:
            value = value_div.text.strip()
            if name:
                data[name] = value
            else:
                title_div = item.find("div", class_="offer__info-title")
                if title_div and "Город" in title_div.text:
                    span = item.find("span")
                    data["город"] = span.text.strip() if span else value

    # dl блоки — "О квартире"
    for dl in soup.findAll("dl"):
        dt = dl.find("dt")
        dd = dl.find("dd")
        if dt and dd and dt.get("data-name"):
            data[dt["data-name"]] = dd.text.strip()

    # Описание - фикс
    desc_div = soup.find("div", class_=lambda c: c and "a-text" in c)
    data["описание"] = desc_div.get_text(separator="\n").strip() if desc_div else None

    return data


# Тест на всех 11 ссылках
test_links = [
    'https://krisha.kz/a/show/1012063292',
    'https://krisha.kz/a/show/1011939022',
    'https://krisha.kz/a/show/1011458594',
    'https://krisha.kz/a/show/1011710667',
    'https://krisha.kz/a/show/1012269199',
    'https://krisha.kz/a/show/1012271793',
    'https://krisha.kz/a/show/1012150112',
    'https://krisha.kz/a/show/1003362629',
    'https://krisha.kz/a/show/689196236',
    'https://krisha.kz/a/show/1010846608',
    'https://krisha.kz/a/show/1009995076',
]

results = []
for url in test_links:
    print(f"Парсим: {url}")
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    result = parse_listing(url, soup)
    results.append(result)
    print(f"  ✅ Комнаты: {result.get('комнаты')}, Цена: {result.get('цена')}, Описание: {'есть' if result.get('описание') else 'нет'}")
    time.sleep(random.uniform(1, 2.5))

print(f"\nГотово! {len(results)} объявлений")

Парсим: https://krisha.kz/a/show/1012063292
  ✅ Комнаты: 3, Цена: 97000000, Описание: нет
Парсим: https://krisha.kz/a/show/1011939022
  ... (ещё 18 строк) ...
  ✅ Комнаты: 4, Цена: 79205400, Описание: есть

Готово! 11 объявлений


## 9. Тест парсера на 11 объявлениях
Проверяем корректность парсинга перед полным запуском.

In [34]:
import pandas as pd

df = pd.DataFrame(results)
print(df.shape)
df

(11, 26)


,url,заголовок,комнаты,цена_raw,цена,город,flat.building,map.complex,house.year,flat.floor,...,ceiling,flat.priv_dorm,has_change,описание,flat.renovation,flat.balcony,flat.door,flat.phone,flat.security,flat.balcony_g
0,https://krisha.kz/a/show/1012063292,"3-комнатная квартира · 120 м² · 6/8 этаж, Арай...",3,97 000 000 〒,97000000,"Алматы, Бостандыкский р-н",монолитный,Сакура,2019,6 из 8,...,3.3 м,нет,Нет,None,NaN,NaN,NaN,NaN,NaN,NaN
1,https://krisha.kz/a/show/1011939022,"1-комнатная квартира · 36 м² · 4/9 этаж, Мухам...",1,23 500 000 〒,23500000,"Астана, Сарайшык р-н",кирпичный,Ansau,2025,4 из 9,...,3 м,нет,Нет,"Пластиковые окна, новая сантехника, тихий двор...",свежий ремонт,балкон,металлическая,есть возможность подключения,"охрана, домофон, видеонаблюдение, видеодомофон",NaN
2,https://krisha.kz/a/show/1011458594,"1-комнатная квартира · 24 м² · 4/4 этаж, мкр №...",1,17 000 000 〒,17000000,"Алматы, Ауэзовский р-н",кирпичный,NaN,1967,4 из 4,...,NaN,да,Нет,None,NaN,балкон,NaN,NaN,NaN,NaN
3,https://krisha.kz/a/show/1011710667,"3-комнатная квартира · 95 м² · 3/19 этаж, Аль-...",3,160 000 000 〒,160000000,"Алматы, Бостандыкский р-н",монолитный,Metropole,2020,3 из 19,...,3 м,нет,Нет,"Пластиковые окна, неугловая, комнаты изолирова...",свежий ремонт,NaN,металлическая,NaN,"домофон, видеонаблюдение",NaN
4,https://krisha.kz/a/show/1012269199,"3-комнатная квартира · 48.2 м² · 2/6 этаж, мкр...",3,34 500 000 〒,34500000,"Алматы, Алатауский р-н",монолитный,Buta Fenomen,2024,2 из 6,...,2.85 м,нет,Нет,Евро трёшка \nПосле ремонта \nТри окна,NaN,NaN,NaN,NaN,NaN,NaN
5,https://krisha.kz/a/show/1012271793,"3-комнатная квартира · 68 м² · 1/5 этаж, Кабан...",3,32 900 000 〒,32900000,"Астана, Есильский р-н",кирпичный,NaN,2017,1 из 5,...,2.7 м,нет,Нет,"Пластиковые окна, комнаты изолированы.\n\nВ Пр...","не новый, но аккуратный ремонт",балкон,металлическая,NaN,NaN,NaN
6,https://krisha.kz/a/show/1012150112,"3-комнатная квартира · 76 м² · 1/9 этаж, мкр А...",3,53 500 000 〒,53500000,"Алматы, Ауэзовский р-н",панельный,NaN,1984,1 из 9,...,2.8 м,нет,Нет,Продаётся уютная квартира с очень удобным расп...,NaN,лоджия,металлическая,нет,"решетки на окнах, домофон, видеонаблюдение",да
7,https://krisha.kz/a/show/1003362629,"5-комнатная квартира · 249.41 м², Аль-Фараби 39",5,из title: 174337590,174337590,"Астана, Есильский р-н",монолитный,Wall Street,2026,NaN,...,3.1 м,NaN,NaN,Готовые квартиры и пентхаусы в клубном доме «W...,NaN,NaN,NaN,NaN,NaN,NaN
8,https://krisha.kz/a/show/689196236,"1-комнатная квартира · 47.01 м², Радостовца 165/1",1,из title: 43249200,43249200,"Алматы, Бостандыкский р-н",монолитный,Tumar,2025,NaN,...,3 м,NaN,NaN,"“TUMAR” – Для всей семьи! Уют, Комфорт, Безопа...",NaN,NaN,NaN,NaN,NaN,NaN
9,https://krisha.kz/a/show/1010846608,"3-комнатная квартира · 62.4 м² · 5/5 этаж, Кло...",3,48 500 000 〒,48500000,"Алматы, Алмалинский р-н",панельный,NaN,1978,5 из 5,...,NaN,нет,Нет,"Пластиковые окна, неугловая, комнаты изолирова...","не новый, но аккуратный ремонт",балкон,металлическая,NaN,"домофон, видеонаблюдение",да


## 10. Основной парсинг, 7,000 объявлений
Запускаем на случайной выборке из 7,000 ссылок. Auto-retry при бане (429), checkpoint каждые 500 записей, логирование всех ошибок.

In [36]:
import random
import time
import re
import pandas as pd

# 7000 случайных ссылок
sample_links = random.sample(all_links_combined, 7000)

def parse_listing(url, soup):
    data = {"url": url}

    h1 = soup.find("h1")
    data["заголовок"] = h1.text.strip() if h1 else None

    if data["заголовок"]:
        rooms_match = re.search(r"(\d+)-комнатная", data["заголовок"])
        data["комнаты"] = int(rooms_match.group(1)) if rooms_match else None
    else:
        data["комнаты"] = None

    price_div = soup.find("div", class_="offer__price")
    if price_div:
        price_raw = price_div.text.strip()
        digits = re.sub(r"[^\d]", "", price_raw)
        data["цена"] = int(digits) if digits else None
    else:
        title_tag = soup.find("title")
        if title_tag:
            price_match = re.search(r"за (\d+)", title_tag.text)
            data["цена"] = int(price_match.group(1)) if price_match else None
        else:
            data["цена"] = None

    for item in soup.findAll("div", class_="offer__info-item"):
        name = item.get("data-name")
        value_div = item.find("div", class_="offer__advert-short-info")
        if value_div:
            value = value_div.text.strip()
            if name:
                data[name] = value
            else:
                title_div = item.find("div", class_="offer__info-title")
                if title_div and "Город" in title_div.text:
                    span = item.find("span")
                    data["город"] = span.text.strip() if span else value

    for dl in soup.findAll("dl"):
        dt = dl.find("dt")
        dd = dl.find("dd")
        if dt and dd and dt.get("data-name"):
            data[dt["data-name"]] = dd.text.strip()

    desc_div = soup.find("div", class_=lambda c: c and "a-text" in c)
    data["описание"] = desc_div.get_text(separator="\n").strip() if desc_div else None

    return data


# Главный loop
results = []
errors = []
SAVE_EVERY = 500  # сохраняем каждые 500 ссылок

for i, url in enumerate(sample_links, 1):
    try:
        response = requests.get(url, headers=headers, timeout=15)

        # Если забанили, ждём дольше и пробуем ещё раз
        if response.status_code == 429:
            print(f"  Бан (429)! Ждём 60 сек...")
            time.sleep(60)
            response = requests.get(url, headers=headers, timeout=15)

        if response.status_code != 200:
            print(f"  [{i}] Ошибка {response.status_code}: {url}")
            errors.append({"url": url, "error": response.status_code})
            continue

        soup = BeautifulSoup(response.content, 'html.parser')
        result = parse_listing(url, soup)
        results.append(result)

        print(f"  [{i}/7000] Комнаты: {result.get('комнаты')}, Цена: {result.get('цена')}, Город: {result.get('город')}")

    except Exception as e:
        print(f"  [{i}] Упало с ошибкой: {e} | {url}")
        errors.append({"url": url, "error": str(e)})

    # Сохраняем каждые 500 ссылок
    if i % SAVE_EVERY == 0:
        df_temp = pd.DataFrame(results)
        df_temp.to_csv(f"krisha_checkpoint_{i}.csv", index=False, encoding="utf-8-sig")
        print(f"\n Сохранено {i} записей → krisha_checkpoint_{i}.csv\n")

    time.sleep(random.uniform(1, 2.5))


# Финальное сохранение
df_final = pd.DataFrame(results)
df_final.to_csv("krisha_final.csv", index=False, encoding="utf-8-sig")

df_errors = pd.DataFrame(errors)
df_errors.to_csv("krisha_errors.csv", index=False, encoding="utf-8-sig")

print(f"\n ГОТОВО!")
print(f" Успешно: {len(results)}")
print(f" Ошибок: {len(errors)}")
print(f" Файл: krisha_final.csv")

  [1/7000] Комнаты: 2, Цена: 25000000, Город: Семей, Красный Кордон
  [2/7000] Комнаты: 3, Цена: 24000000, Город: Астана, Сарыарка р-н
  [3/7000] Комнаты: 3, Цена: 51999999, Город: Астана, Алматы р-н
  ... (ещё 7041 строк) ...
 Успешно: 7000
 Ошибок: 0
 Файл: krisha_final.csv
